# Семинар 11. Деревья и кучи

На практике разберём одну иерархию расходов тремя способами, соберём небольшое бинарное дерево поиска и закончим очередью обращений. Во всех упражнениях сначала называем инвариант и только потом пишем цикл или рекурсию.

## Цели

После семинара вы сможете:

- читать и строить небольшие деревья объектов;
- трассировать рекурсивный и итеративный DFS;
- получать уровни дерева через BFS;
- восстанавливать путь до найденного узла;
- вставлять и искать ключи в BST;
- проверять вырождение несбалансированного дерева;
- выполнять операции min-heap и max-heap;
- реализовывать стабильную очередь с приоритетом и top-k.

## Перед началом

Нужен Python 3.14: упражнения с `heappush_max` и `heappop_max` используют API, появившийся именно в этой версии. Работа рассчитана примерно на 90 минут: 30 минут на обходы, 25 минут на BST, 25 минут на кучи, 10 минут на обсуждение результатов и вопросы. Проверяйте решения небольшими `assert`, не вводом с клавиатуры.

## Общие данные: категории семейных расходов

Сумма может находиться и во внутренней категории, и в листе. Это не файловая система и не абстрактная буква `x`, а упрощённая модель отчёта, который вполне можно получить из банковских данных.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Category:
    name: str
    amount: int = 0
    children: list["Category"] = field(default_factory=list)

expenses = Category("all", 100, [
    Category("food", children=[
        Category("groceries", 3200),
        Category("cafes", 1800),
    ]),
    Category("transport", children=[
        Category("metro", 600),
        Category("taxi", 900),
    ]),
    Category("rent", 35_000),
])

## Упражнение 1. Сумма и число узлов

Реализуйте две рекурсивные функции. До запуска ответьте: что будет базовым случаем, сколько раз посетится каждый узел и сколько вызовов одновременно останется в стеке? Пустого дерева в этой модели нет: функция всегда получает существующий корень.

In [ ]:
def total_amount(node: Category) -> int:
    ...

def count_nodes(node: Category) -> int:
    ...

assert total_amount(expenses) == 41_600
assert count_nodes(expenses) == 8
assert total_amount(Category("empty leaf")) == 0

## Упражнение 2. Путь до категории через DFS

Верните список имён от корня до первого узла с нужным именем. Если имя не найдено, верните `None`. Не храните путь в глобальном изменяемом списке: каждый вызов должен возвращать собственный результат ветви.

Полезная схема: проверить текущий узел, затем спросить каждого ребёнка; успешный путь дополнить текущим именем и немедленно вернуть вверх.

In [ ]:
def find_path(node: Category, target: str) -> list[str] | None:
    ...

assert find_path(expenses, "taxi") == ["all", "transport", "taxi"]
assert find_path(expenses, "all") == ["all"]
assert find_path(expenses, "travel") is None

## Упражнение 3. Итеративный DFS

Соберите имена в том же preorder-порядке, что и рекурсивная функция: текущий узел, затем дети слева направо. После каждого шага на бумаге записывайте содержимое стека. Проверьте отдельно, что произойдёт без `reversed`.

In [ ]:
def preorder_iterative(root: Category) -> list[str]:
    result = []
    stack = [root]
    while stack:
        # извлеките узел, сохраните имя и положите детей
        ...
    return result

assert preorder_iterative(expenses) == [
    "all", "food", "groceries", "cafes",
    "transport", "metro", "taxi", "rent",
]

## Упражнение 4. Уровни через BFS

Верните словарь `глубина -> имена на этом уровне`. Используйте `deque`, в очереди храните пару `(node, depth)`. Сравните максимум размера очереди с максимумом размера стека в предыдущем упражнении именно на этом дереве.

In [ ]:
from collections import defaultdict, deque

def group_by_depth(root: Category) -> dict[int, list[str]]:
    levels = defaultdict(list)
    queue = deque([(root, 0)])
    while queue:
        ...
    return dict(levels)

assert group_by_depth(expenses) == {
    0: ["all"],
    1: ["food", "transport", "rent"],
    2: ["groceries", "cafes", "metro", "taxi"],
}

## Бинарное дерево поиска: индекс по сумме

Для учебного примера построим индекс `сумма -> описание`. В реальной программе одинаковые суммы потребовали бы список значений или составной ключ; здесь ключи уникальны, чтобы сосредоточиться на структуре. Перед каждой вставкой проговаривайте: меньше — налево, больше — направо.

In [ ]:
@dataclass
class SearchNode:
    key: int
    value: str
    left: "SearchNode | None" = None
    right: "SearchNode | None" = None

## Упражнение 5. Вставка и поиск в BST

Реализуйте операции итеративно. Пустая ссылка означает место для нового узла или отсутствие ключа. При повторном ключе обновите значение. Не обещайте `O(log N)` без условия о высоте.

In [ ]:
def bst_insert(root: SearchNode | None, key: int, value: str) -> SearchNode:
    ...

def bst_find(root: SearchNode | None, key: int) -> str | None:
    ...

root = None
for key, value in [(900, "taxi"), (600, "metro"), (1800, "cafes"), (3200, "groceries")]:
    root = bst_insert(root, key, value)

assert bst_find(root, 1800) == "cafes"
assert bst_find(root, 1000) is None
root = bst_insert(root, 900, "taxi updated")
assert bst_find(root, 900) == "taxi updated"

## Упражнение 6. Inorder и вырождение

Сначала верните пары из BST по возрастанию ключа. Затем напишите `height`, считая высоту листа равной нулю, а пустого дерева — `-1`. Постройте дерево из ключей `1..10` по порядку и объясните результат без ссылки на то, что «рекурсия медленная».

In [ ]:
def bst_items(root: SearchNode | None) -> list[tuple[int, str]]:
    ...

def height(root: SearchNode | None) -> int:
    ...

assert [key for key, _ in bst_items(root)] == [600, 900, 1800, 3200]

skewed = None
for key in range(1, 11):
    skewed = bst_insert(skewed, key, str(key))
assert height(skewed) == 9

## Min-heap: следующая ближайшая дата

Календарный план естественно использует min-heap: раньше наступающая дата должна быть наверху. Здесь даты уже представлены порядковыми номерами дней, чтобы не отвлекаться на парсинг. Обратите внимание: `heapify` перестраивает исходный список на месте.

## Упражнение 7. Операции min-heap

Превратите список в кучу, добавьте событие и извлеките все события в хронологическом порядке. Не заменяйте упражнение вызовом `sorted`: цель — увидеть, что только последовательные извлечения создают полный порядок.

In [ ]:
import heapq

events = [(12, "rent"), (5, "internet"), (20, "tax")]
# heapify, добавление события (8, "salary") и последовательное извлечение
...

assert ordered == [
    (5, "internet"), (8, "salary"), (12, "rent"), (20, "tax")
]

## Max-heap: самое срочное обращение

> **Новое в Python 3.14.** `heapq.heappush_max` и `heapq.heappop_max` позволяют хранить обычный положительный приоритет. В старых версиях тот же пример потребовал бы min-heap и `-priority`.

Чтобы одинаково срочные обращения обслуживались по поступлению, запись имеет вид `(priority, -order, id)`. Уникальный `order` не даёт сравнению дойти до самого объекта обращения.

## Упражнение 8. Стабильная очередь обращений

Реализуйте функцию на новом max-heap API. Проверьте два одинаковых приоритета и не изменяйте исходные словари. Затем устно перепишите только ключ записи для старого min-heap варианта.

In [ ]:
from heapq import heappop_max, heappush_max

def priority_order(requests: list[dict]) -> list[str]:
    ...

requests = [
    {"id": "r-1", "priority": 2},
    {"id": "r-2", "priority": 5},
    {"id": "r-3", "priority": 5},
    {"id": "r-4", "priority": 1},
]
assert priority_order(requests) == ["r-2", "r-3", "r-1", "r-4"]

## Упражнение 9. Три крупнейшие операции

Поток сумм может быть намного больше памяти, поэтому нельзя сортировать или сохранять его целиком. Поддерживайте min-heap размера не более `k`. Корень — наименьший из текущих победителей; заменяйте его, когда приходит более крупная сумма. Результат из `k` элементов в конце разрешено отсортировать.

In [ ]:
from collections.abc import Iterable

def top_k_amounts(amounts: Iterable[int], k: int) -> list[int]:
    ...

assert top_k_amounts([120, 900, 450, 1500, 700, 2000], 3) == [2000, 1500, 900]
assert top_k_amounts([5, 4], 0) == []
assert top_k_amounts(iter([5, 4]), 5) == [5, 4]

## Обсуждение перед сдачей

Для каждой функции ответьте:

- какой инвариант должен быть истинным после каждой итерации;
- от чего зависит память: от числа узлов, высоты, ширины или `k`;
- изменяется ли входной объект;
- что произойдёт на пустом входе, листе и при равных приоритетах;
- какая часть решения зависит именно от Python 3.14.

Если ответ нельзя сформулировать без чтения кода по строкам, решение стоит упростить.

## Самопроверка

1. Какой базовый случай был у суммы дерева?
2. Почему путь DFS надо возвращать вверх, а не хранить глобально?
3. Как стек меняет порядок детей?
4. Что одновременно лежит в очереди BFS?
5. Почему отсортированные вставки портят высоту простого BST?
6. Каким обходом BST получается сортировка?
7. Почему `heapify` быстрее последовательной сборки кучи?
8. Что гарантирует корень min-heap?
9. Какие функции упражнения 8 требуют Python 3.14?
10. Почему порядок записан как отрицательное число в max-heap?
11. Почему для top-k крупнейших внутри используется min-heap?
12. В какой задаче `dict` был бы разумнее нашего BST?

## Итоги

- Рекурсивный код естественно повторяет структуру дерева, а явный стек снимает ограничение глубины.
- DFS и BFS посещают те же узлы, но дают разный порядок и профиль памяти.
- Скорость BST определяется фактической высотой.
- Куча хранит ровно столько порядка, сколько нужно для быстрого экстремума.
- Python 3.14 даёт прямые операции max-heap; стабильность очереди по-прежнему проектируем сами через счётчик.
- Min-heap размера `k` хранит только текущих победителей и подходит для потока данных.

Домашняя работа продолжает эти идеи: путь по дереву, BST, top-k и необязательный изменяемый планировщик.